In [ ]:
"""
================================================================================
TESIS MAGÍSTER EN ECONOMÍA UC
Contagio predictivo sectorial y NBFIs en Chile
================================================================================
Script: MI_HE.ipynb
Autor:  Carlos González
Fecha:  Abril 2026
Ultima actualizacion: Mayo 2026

FIGURAS Sección 3. Marco Institucional y Hechos Estilizados:
 3.A. El sistema financiero chileno en perspectiva comparada
    Fig 1_2  -> Activos por sector (niveles y % sociedades financieras) -> Apéndice
    Fig 2 -> Activos como proporción del PIB (all sectors) desde matriz de interconexión sin considerar dinero legal -> Incluido en MI-HE
    fig 2_2 -> Activos como proporción del PIB (all sectors) -> Apéndice
    fig 3 (FALTA) -> Instrumentos de cada sector sectores desde matriz de interconexión sin considerar dinero legal -> incluido en MI-HE
        Panel 3 filas x 4 comunas = 12 paneles. Por el lado de los activos.
    fig 3_2 -> " Por el lado de los pasivos

 3.B. Tres episodios de estrés
    GCF, crisis social y retiros

 3.C. La singularidad de la interconexión NBFI – Bancos
    fig 3 -> Exposición bilateral NBFI -> Bancos vs Bancos -> NBFI (asimetría)
    fig 4 -> Exposición por sector NBFI → Bancos (heterogeneidad)
    fig 3_1 -> 

 3.D. La pregunta de diversificación: ¿efectiva o ilusoria?
    fig 5 -> HHI fuentes financiamiento sector real
    fig 6 -> Crédito al sector real por tipo de acreedor

NOTA METODOLÓGICA — PIB (Fig 2):
  Se utilizan los valores del PIB nominal anual publicados por el Banco Central
  de Chile (BDE, series de Cuentas Nacionales Trimestrales a precios corrientes).
  El ratio activo/PIB se construye dividiendo el stock de activos del período t
  por el PIB nominal anualizado por suma movil correspondiente. Esta convención es estándar en
  la literatura de NBFI (FSB, 2023, 2025) y permite comparación internacional.

Fuentes: 
    - 1_clean_data
     Figuras 1_2 y 2:
        - panel_balances.csv
        - panel_matriz_saldos.csv

     Figuras 3-5
        - panel_exp_nbfi_bancos_agregado
        - panel_exp_nbfi_bancos_sector
        - panel_exp_bancos_nbfi
        - panel_is_upper_worms

     Figuras 6-8
        - resumen_hhi_por_periodo
        - panel_red_saldos
        
        
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURACIÓN DE RUTAS
# ============================================================

BASE_DIR  = Path("../..")
DATA_DIR  = BASE_DIR / "1_datos/1_clean_data"
FIG_DIR   = BASE_DIR / "3_resultados/3_resultados_he_red/figuras/1_MI_HE"
#TABLE_DIR = BASE_DIR / "3_resultados/3_resultados_he_red/tablas"

for d in [FIG_DIR]: #for d in [FIG_DIR;TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# PALETA Y ETIQUETAS
# ============================================================

COLORS = {
    "S121":      "#2C3E50",
    "S122":      "#2980B9",
    "S123":      "#27AE60",
    "S124":      "#166757",
    "S125_S126": "#8E44AD",
    "S128":      "#E67E22",
    "S129":      "#AA4B41",
    "S13":       "#7F8C8D",
    "S11":       "#613F06",
    "S14_S15":   "#1ABC9C",
    "S2":        "#74D5E4",
    "NBFI":     "#E74C3C",  
}

COLORES_INSTR = {
    "Depositos":    "#27AE60",  # verde
    "Prestamos":    "#2980B9",  # azul
    "Titulos deuda":"#E74C3C",  # rojo 
    "Cuotas FMM":   "#8E44AD",  # violeta
    "Cuotas FNM":   "#E67E22",  # naranja
}


LABELS = {
    "S121":      "Banco Central",
    "S122":      "Bancos",
    "S123":      "FMM",
    "S124":      "FNM",
    "S125_S126": "Otros intermediarios",
    "S128":      "Seguros",
    "S129":      "AFP",
    "S13":       "Gobierno",
    "S11":       "Empresas NF",
    "S14_S15":   "Hogares",
    "S2":        "Resto del mundo",
    "NBFI":     "NBFI",
}

NBFI         = ["S123", "S124", "S125_S126", "S128", "S129"]
NBFI_ACOTADO = ["S123", "S124", "S125_S126"]
BANCARIO     = ["S122"]
BANCO_CENTRAL = ["S121"]
GOBIERNO      = ["S13"]
SECTOR_REAL    = ["S11", "S14_S15"]
HOGARES = ["S14_S15"]
EMPRESAS = ["S11"]
RESTO_MUNDO      = ["S2"]

# ============================================================
# PIB NOMINAL TRIMESTRAL (fuente: BCCh, Cuentas Nacionales Trimestrales)
# Miles de millones de pesos corrientes, precios corrientes de cada período
# ============================================================
# Se parte desde 2002,2 porque haré suma movil para graficar respecto al PIB.
PIB_TRIMESTRAL = {
    (2002,2):11838.83,(2002,3):11681.80,(2002,4):13083.87,
    (2003,1):12939.08,(2003,2):13092.71,(2003,3):12696.06,(2003,4):14169.49,
    (2004,1):14291.96,(2004,2):14801.96,(2004,3):14800.11,(2004,4):16497.72,
    (2005,1):16309.90,(2005,2):16711.53,(2005,3):16688.90,(2005,4):18757.61,
    (2006,1):19251.55,(2006,2):20563.29,(2006,3):20038.60,(2006,4):21724.09,
    (2007,1):21946.86,(2007,2):22597.56,(2007,3):21613.92,(2007,4):24001.14,
    (2008,1):24088.41,(2008,2):23898.72,(2008,3):21977.66,(2008,4):23902.33,
    (2009,1):23247.62,(2009,2):23481.16,(2009,3):22980.23,(2009,4):26429.46,
    (2010,1):25685.17,(2010,2):26801.71,(2010,3):27399.25,(2010,4):30891.74,
    (2011,1):29932.27,(2011,2):29967.72,(2011,3):28904.55,(2011,4):32704.77,
    (2012,1):31759.25,(2012,2):32084.08,(2012,3):31078.56,(2012,4):35051.50,
    (2013,1):33443.42,(2013,2):33910.58,(2013,3):32986.24,(2013,4):36968.95,
    (2014,1):36032.62,(2014,2):36555.82,(2014,3):35389.85,(2014,4):39973.00,
    (2015,1):39273.33,(2015,2):39194.30,(2015,3):37973.39,(2015,4):42181.88,
    (2016,1):42096.38,(2016,2):41215.11,(2016,3):40591.43,(2016,4):44861.77,
    (2017,1):43580.31,(2017,2):43880.00,(2017,3):43577.87,(2017,4):48276.72,
    (2018,1):47046.22,(2018,2):46992.93,(2018,3):45024.22,(2018,4):50371.50,
    (2019,1):48589.90,(2019,2):48590.64,(2019,3):47342.74,(2019,4):51024.88,
    (2020,1):51190.92,(2020,2):46170.51,(2020,3):47398.30,(2020,4):56546.78,
    (2021,1):56150.20,(2021,2):57036.79,(2021,3):59138.22,(2021,4):67231.06,
    (2022,1):63770.24,(2022,2):63480.77,(2022,3):63961.68,(2022,4):71741.55,
    (2023,1):70236.09,(2023,2):68963.52,(2023,3):67633.19,(2023,4):75263.85,
    (2024,1):76699.67,(2024,2):74885.95,(2024,3):74738.16,(2024,4):84357.40,
    (2025,1):81964.17,(2025,2):82873.23,(2025,3):82692.58,(2025,4):92447.74,
}

# ============================================================
# EPISODIOS DE ESTRÉS
# ============================================================

EPISODIOS_LINEAS = {
    "GFC\n2008T3":       "2008T3",
    "Estallido\n2019T4": "2019T4",
}

# ============================================================
# ESTILO GLOBAL
# ============================================================

plt.rcParams.update({
    "font.family":       "serif",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "legend.fontsize":   8,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
})

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def tp(p):
    y, q = p.split("T")
    return int(y) + (int(q) - 1) / 4

def xax(ax):
    ax.set_xlim(2002.5, 2025.8)
    ax.set_xticks([2004, 2008, 2012, 2016, 2020, 2024])
    ax.set_xticklabels(["2004", "2008", "2012", "2016", "2020", "2024"], rotation=0)

def vl(ax):
    """Marca los tres episodios de estrés en el gráfico."""
    for lbl, p in EPISODIOS_LINEAS.items():
        ax.axvline(tp(p), color="gray", ls=":", lw=1.2, alpha=0.7)
        ax.text(tp(p) + 0.1, 0.93, lbl, fontsize=7, color="dimgray",
                va="top", transform=ax.get_xaxis_transform())
    t_ini = tp("2020T3")
    t_fin = tp("2021T2")
    ax.axvspan(t_ini, t_fin, color="gray", alpha=0.15, zorder=0)
    ax.text((t_ini + t_fin) / 2, 0.02, "Retiros\nAFP", fontsize=7,
            color="dimgray", ha="center", va="bottom",
            transform=ax.get_xaxis_transform())

def legend_below(ax, ncol=3):
    """Leyenda centrada debajo de cada gráfico, fuera del área de datos."""
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22),
              fontsize=8, frameon=True, framealpha=0.9,
              edgecolor="silver", ncol=ncol)

def save_fig(fig, name):
    fig.tight_layout(rect=[0, 0, 1, 0.93], h_pad=4.0, w_pad=2.0)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    ✓ {name}")

def build_pib_df():
    """
    Construye el denominador del PIB para el ratio activos/PIB.

    Metodología:
    Se usa el PIB nominal trimestral a precios corrientes (BCCh CNT, tabla 2.1)
    y se calcula una media móvil de suma de 4 trimestres consecutivos:

        PIB_MM4_t = PIB_t + PIB_{t-1} + PIB_{t-2} + PIB_{t-3}

    Esta suma corresponde al PIB nominal de los últimos 12 meses (trailing),
    que es el denominador correcto para un ratio stock (trimestral) / flujo.
    Evita el salto discreto que produce dividir por el PIB del año calendario
    completo, y es consistente con la práctica del FSB (2023, 2025) para
    calcular activos NBFI / PIB en comparaciones internacionales.

    Los primeros 3 trimestres (sin ventana completa) quedan como NaN
    y no se grafican.
    """
    rows = [
        {"periodo": f"{y}T{q}", "año": y, "trimestre": q, "pib_trim": v}
        for (y, q), v in PIB_TRIMESTRAL.items()
    ]
    df = pd.DataFrame(rows).sort_values("periodo").reset_index(drop=True)
    # Suma móvil de 4 trimestres consecutivos (trailing 12 meses)
    df["pib_mm4"] = df["pib_trim"].rolling(window=4, min_periods=4).sum()
    return df[["periodo", "pib_trim", "pib_mm4"]]

def build_totales_nacionales(bal):
    """
    Total de activos financieros nacionales (excluye S2 = Resto del Mundo):
      - total_nac:    incluye acciones y participaciones de capital
      - total_nac_sa: excluye acciones (solo depósitos + títulos + préstamos + cuotas)

    La exclusión de acciones es relevante porque S125_S126 (Otros intermediarios)
    tiene ~96% de su activo en acciones de sociedades de cartera, lo que distorsiona
    su tamaño como intermediario financiero frente al resto de los sectores.
    """
    bal["act_sin_acciones"] = (
        bal["act_titulos_total_calc"].fillna(0) +
        bal["act_prestamos_total_calc"].fillna(0) +
        bal["act_depositos_total_calc"].fillna(0) +
        bal["act_cuotas_fmm"].fillna(0) +
        bal["act_cuotas_fnm"].fillna(0)
    )
    bal_res = bal[bal.sector_codigo != "S2"]
    tn = bal_res.groupby("periodo")["activo_proxy"].sum().reset_index()
    tn.columns = ["periodo", "total_nac"]
    tn["t"] = tn["periodo"].apply(tp)
    tn_sa = bal_res.groupby("periodo")["act_sin_acciones"].sum().reset_index()
    tn_sa.columns = ["periodo", "total_nac_sa"]
    tn_sa["t"] = tn_sa["periodo"].apply(tp)
    return tn, tn_sa


# ============================================================
# FIG 1: ACTIVOS POR SECTOR NIVEL Y PROPORCIÓN DE SF
# ============================================================

def fig1_activos_por_sector(bal):
    print("  Fig 1: Activos por sector...")
    bal = bal.copy()
    bal["activo_proxy"] = bal["act_total"].fillna(bal["pas_total"])
    sectores_plot = ["S122", "S129", "S128", "S124", "S123", "S125_S126"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Activos financieros del sistema financiero chileno, 2003–2025",
                 fontsize=11)

    ax = axes[0]
    for sec in sectores_plot:
        sub = bal[bal.sector_codigo == sec].sort_values("t")
        ax.plot(sub["t"], sub["activo_proxy"] / 1000,
                color=COLORS[sec], label=LABELS[sec], lw=1.8)
    xax(ax); vl(ax)
    ax.set_title("A. Niveles (miles de MMCL)")
    ax.set_ylabel("Miles de MMCL (billones escala larga)")
    legend_below(ax, ncol=3)

    ax2 = axes[1]
    total_sf = bal[bal.sector_codigo.isin(BANCARIO + NBFI)].groupby(
        "periodo")["activo_proxy"].sum().reset_index()
    total_sf.columns = ["periodo", "total"]
    total_sf["t"] = total_sf["periodo"].apply(tp)
    for sec in sectores_plot:
        sub = bal[bal.sector_codigo == sec][["periodo", "t", "activo_proxy"]].merge(
            total_sf, on=["periodo", "t"])
        sub["sh"] = sub["activo_proxy"] / sub["total"] * 100
        ax2.plot(sub.sort_values("t")["t"], sub.sort_values("t")["sh"],
                 color=COLORS[sec], label=LABELS[sec], lw=1.8)
    xax(ax2); vl(ax2)
    ax2.set_title("B. Participación (% del sistema financiero: bancos + NBFI)")
    ax2.set_ylabel("% del sistema financiero")
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
    legend_below(ax2, ncol=3)

    save_fig(fig, "fig1_1_activos_sector")

# ============================================================
# FIG 2: ACTIVOS COMO PROPORCIÓN DEL PIB (ALL SECTORS)
# ============================================================

def fig2_activos_pib(bal):
    """
    Ratio activos/PIB usando PIB nominal trailing 4 trimestres (suma móvil).

    Denominador: suma de los PIB nominales de los 4 trimestres anteriores
    incluyendo el actual (trailing 12 meses). Esto produce una serie continua
    sin saltos discretos de año en año, adecuada para comparar con el stock
    trimestral de activos financieros.

    Fuente PIB: BCCh, Cuentas Nacionales Trimestrales (tabla 2.1),
    precios corrientes de cada período.
    """
    print("  Fig 11: Activos como % del PIB (suma móvil 4 trimestres)...")

    bal = bal.copy()
    bal["activo_proxy"] = bal["act_total"].fillna(bal["pas_total"])

    pib_df = build_pib_df()
    bal = bal.merge(pib_df[["periodo", "pib_mm4"]], on="periodo", how="left")

    sectores_plot = ["S122", "S129", "S128", "S124", "S123", "S125_S126"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        "Tenencia de activos del sistema financiero como proporción del PIB, 2003–2025\n"
        "(activos / PIB nominal anualizado suma móvil 4 trimestres)",
        fontsize=11)

    # Panel A: cada sector como % PIB MM4
    ax = axes[0]
    for sec in sectores_plot:
        sub = bal[(bal.sector_codigo == sec) & bal["pib_mm4"].notna()].sort_values("t")
        ax.plot(sub["t"], sub["activo_proxy"] / sub["pib_mm4"] * 100,
                color=COLORS[sec], label=LABELS[sec], lw=1.8)
    xax(ax); vl(ax)
    ax.set_title("A. Activos en sector Bancario y NBFIs (% del PIB)")
    ax.set_ylabel("% del PIB")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_yticks(np.arange(0, 150, 20))
    legend_below(ax, ncol=3)

    # Panel B: NBFI total, Bancos y SF total como % PIB MM4
    ax2 = axes[1]
    for grp, label, color, ls in [
        (NBFI,            "NBFI total",             COLORS["NBFI"], "-"),
        (BANCARIO,        "Bancos",                 COLORS["S122"], "-"),
        (GOBIERNO, "Gobierno", COLORS["S13"],    "-"),
        (HOGARES, "Hogares", COLORS["S14_S15"], "-"),
        (EMPRESAS, "Empresas", COLORS["S11"], "-"),
        (RESTO_MUNDO, "Resto del Mundo", COLORS["S2"], "-"),
        (BANCO_CENTRAL, "Banco Central", COLORS["S121"], "-")
    ]:
        agg = bal[bal.sector_codigo.isin(grp)].groupby(
            ["periodo", "t", "pib_mm4"])["activo_proxy"].sum().reset_index()
        agg = agg[agg["pib_mm4"].notna()].sort_values("t")
        ax2.plot(agg["t"], agg["activo_proxy"] / agg["pib_mm4"] * 100,
                 color=color, label=label, lw=2.2, ls=ls)
    xax(ax2); vl(ax2)
    ax2.set_title("B. Sistema financiero total (% del PIB)")
    ax2.set_ylabel("% del PIB")
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax2.set_yticks(np.arange(0, 360, 50))
    legend_below(ax2, ncol=3)

    save_fig(fig, "fig2_activos_pib")

# ============================================================
# FIG 2B: ACTIVOS DESDE RED WHO-TO-WHOM COMO % DEL PIB
# (sin Dinero legal — equivalente a fig2 pero desde la red)
# ============================================================
#El activo de cada sector en la red es lo que ese sector tiene invertido en otros sectores, 
# no lo que otros tienen invertido en él. Para el sector real (Hogares y Empresas) ese activo
#  en la red es bajo porque hogares y empresas no intermedian mucho. Para el Resto del Mundo
#  también puede ser bajo o alto dependiendo de cuánto invierte en Chile.
def fig2b_activos_red_pib():
    """
    Mismo layout que fig2_activos_pib pero el numerador viene
    de panel_red_saldos.csv (activos de cada sector en la red,
    excluyendo Dinero legal por construcción del archivo).

    Denominador: PIB nominal suma móvil 4 trimestres (idéntico a fig2).

    Diferencia respecto a fig2:
      - Excluye acciones, reservas técnicas y otras cuentas
        que sí están en el balance pero no en la red
      - Incluye solo los 5 instrumentos de intermediación:
        Préstamos, Títulos deuda, Depósitos, Cuotas FMM, Cuotas FNM
      - Numerador y denominador conceptualmente consistentes
        con los indicadores IS, HHI y exposición bilateral
    """
    print("  Fig 2B: Activos desde red who-to-whom como % del PIB...")

    df_red = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df_red["t"] = df_red["periodo"].apply(tp)

    pib_df = build_pib_df()

    # Activo de cada sector = suma de sus exposiciones activas en la red
    # (lo que tiene invertido en todos los sectores en los 5 instrumentos)
    act_sector = df_red.groupby(
        ["periodo","t","sector_activo_codigo"]
    )["valor"].sum().reset_index().rename(
        columns={"sector_activo_codigo":"sector_codigo",
                 "valor":"activo_red"})
    act_sector = act_sector.merge(pib_df[["periodo","pib_mm4"]],
                                  on="periodo", how="left")
    act_sector["activo_pib"] = act_sector["activo_red"] / act_sector["pib_mm4"] * 100
    act_sector = act_sector[act_sector["pib_mm4"].notna()]

    sectores_plot = ["S129","S128","S124","S123","S125_S126"]

    fig, axes = plt.subplots(2, 1, figsize=(10,13))
    fig.suptitle(
        "Activos de intermediación como proporción del PIB, 2003–2025\n"
        "(desde matriz who-to-whom | instrumentos: depósitos, préstamos, "
        "títulos, cuotas FMM y FNM | excluye dinero legal)",
        fontsize=11
    )

    # ── Panel A: por sector bancario y NBFIs ──────────────────────────
    ax = axes[1]
    for sec in sectores_plot:
        sub = act_sector[
            act_sector["sector_codigo"] == sec
        ].sort_values("t")
        if sub.empty: continue
        ax.plot(sub["t"], sub["activo_pib"],
                color=COLORS[sec], label=LABELS[sec], lw=1.8)

    ax.set_title("B. Sectores NBFIs (% del PIB)")
    ax.set_ylabel("% del PIB")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_yticks(np.arange(0, 110, 10))
    xax(ax); vl(ax)
    legend_below(ax, ncol=3)

    # ── Panel B: por grupo ────────────────────────────────────────────
    ax2 = axes[0]

    GRUPOS_B = [
        (NBFI,          "NBFI total",       COLORS["NBFI"], "-",  2.2),
        (BANCARIO,      "Bancos",            COLORS["S122"], "-",  2.2),
        (GOBIERNO,      "Gobierno",          COLORS["S13"],  "--", 1.6),
        (HOGARES,       "Hogares",           COLORS["S14_S15"], "--", 1.6),
        (EMPRESAS,      "Empresas",          COLORS["S11"],  "--", 1.4),
        (RESTO_MUNDO,   "Resto del Mundo",   COLORS["S2"],   "--",  1.4),
        (BANCO_CENTRAL, "Banco Central",     COLORS["S121"], "-",  1.4),
    ]

    for grp, label, color, ls, lw in GRUPOS_B:
        sub = act_sector[
            act_sector["sector_codigo"].isin(grp)
        ].groupby(["periodo","t","pib_mm4"])["activo_red"].sum().reset_index()
        sub = sub[sub["pib_mm4"].notna()].sort_values("t")
        sub["activo_pib"] = sub["activo_red"] / sub["pib_mm4"] * 100
        ax2.plot(sub["t"], sub["activo_pib"],
                 color=color, label=label, lw=lw, ls=ls)

    ax2.set_title("A. Sectores Financieros Institucionales (% del PIB)")
    ax2.set_ylabel("% del PIB")
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax2.set_yticks(np.arange(0, 210, 20))
    xax(ax2); vl(ax2)
    legend_below(ax2, ncol=3)

    save_fig(fig, "fig2b_activos_red_pib_2x1")

# ============================================================
# FIG 2.2B: ACTIVOS DESDE RED WHO-TO-WHOM COMO % DEL PIB
# (sin Dinero legal — equivalente a fig2 pero desde la red)
# ============================================================
#El activo de cada sector en la red es lo que ese sector tiene invertido en otros sectores, 
# no lo que otros tienen invertido en él. Para el sector real (Hogares y Empresas) ese activo
#  en la red es bajo porque hogares y empresas no intermedian mucho. Para el Resto del Mundo
#  también puede ser bajo o alto dependiendo de cuánto invierte en Chile.
def fig2b_activos_red_pib_1x2():
    """
    Mismo layout que fig2_activos_pib pero el numerador viene
    de panel_red_saldos.csv (activos de cada sector en la red,
    excluyendo Dinero legal por construcción del archivo).

    Denominador: PIB nominal suma móvil 4 trimestres (idéntico a fig2).

    Diferencia respecto a fig2:
      - Excluye acciones, reservas técnicas y otras cuentas
        que sí están en el balance pero no en la red
      - Incluye solo los 5 instrumentos de intermediación:
        Préstamos, Títulos deuda, Depósitos, Cuotas FMM, Cuotas FNM
      - Numerador y denominador conceptualmente consistentes
        con los indicadores IS, HHI y exposición bilateral
    """
    print("  Fig 2B: Activos desde red who-to-whom como % del PIB...")

    df_red = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df_red["t"] = df_red["periodo"].apply(tp)

    pib_df = build_pib_df()

    # Activo de cada sector = suma de sus exposiciones activas en la red
    # (lo que tiene invertido en todos los sectores en los 5 instrumentos)
    act_sector = df_red.groupby(
        ["periodo","t","sector_activo_codigo"]
    )["valor"].sum().reset_index().rename(
        columns={"sector_activo_codigo":"sector_codigo",
                 "valor":"activo_red"})
    act_sector = act_sector.merge(pib_df[["periodo","pib_mm4"]],
                                  on="periodo", how="left")
    act_sector["activo_pib"] = act_sector["activo_red"] / act_sector["pib_mm4"] * 100
    act_sector = act_sector[act_sector["pib_mm4"].notna()]

    sectores_plot = ["S129","S128","S124","S123","S125_S126"]

    fig, axes = plt.subplots(1, 2, figsize=(13,5))
    fig.suptitle(
        "Activos de intermediación como proporción del PIB, 2003–2025\n"
        "(desde matriz who-to-whom | instrumentos: depósitos, préstamos, "
        "bonos y cuotas fondos mutuos)",
        fontsize=11
    )

    # ── Panel A: por sector bancario y NBFIs ──────────────────────────
    ax = axes[1]
    for sec in sectores_plot:
        sub = act_sector[
            act_sector["sector_codigo"] == sec
        ].sort_values("t")
        if sub.empty: continue
        ax.plot(sub["t"], sub["activo_pib"],
                color=COLORS[sec], label=LABELS[sec], lw=1.8)

    ax.set_title("B. Sectores NBFIs (% del PIB)")
    ax.set_ylabel("% del PIB")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_yticks(np.arange(0, 110, 10))
    xax(ax); vl(ax)
    legend_below(ax, ncol=3)

    # ── Panel B: por grupo ────────────────────────────────────────────
    ax2 = axes[0]

    GRUPOS_B = [
        (NBFI,          "NBFI total",       COLORS["NBFI"], "-",  2.2),
        (BANCARIO,      "Bancos",            COLORS["S122"], "-",  2.2),
        (GOBIERNO,      "Gobierno",          COLORS["S13"],  "--", 1.6),
        (HOGARES,       "Hogares",           COLORS["S14_S15"], "--", 1.6),
        (EMPRESAS,      "Empresas",          COLORS["S11"],  "--", 1.4),
        (RESTO_MUNDO,   "Resto del Mundo",   COLORS["S2"],   "--",  1.4),
        (BANCO_CENTRAL, "Banco Central",     COLORS["S121"], "-",  1.4),
    ]

    for grp, label, color, ls, lw in GRUPOS_B:
        sub = act_sector[
            act_sector["sector_codigo"].isin(grp)
        ].groupby(["periodo","t","pib_mm4"])["activo_red"].sum().reset_index()
        sub = sub[sub["pib_mm4"].notna()].sort_values("t")
        sub["activo_pib"] = sub["activo_red"] / sub["pib_mm4"] * 100
        ax2.plot(sub["t"], sub["activo_pib"],
                 color=color, label=label, lw=lw, ls=ls)

    ax2.set_title("A. Sectores Financieros Institucionales (% del PIB)")
    ax2.set_ylabel("% del PIB")
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax2.set_yticks(np.arange(0, 210, 20))
    xax(ax2); vl(ax2)
    legend_below(ax2, ncol=3)

    save_fig(fig, "fig2b_activos_red_pib_1x2")
# ============================================================
# FIG 2C: COMPOSICIÓN DE ACTIVOS POR SECTOR (DESDE RED)
# ============================================================

def fig2c_composicion_instrumentos():
    """
    Composición porcentual de activos de intermediación por sector
    institucional, construida desde panel_red_saldos.csv.

    Cada panel muestra qué fracción del activo total del sector
    en la red está en cada instrumento (préstamos, títulos de
    deuda, depósitos, cuotas FNM, cuotas FMM).

    Excluye dinero legal y acciones por construcción de la red.
    El denominador es el activo total del sector en la red
    (misma fuente que los ratios de exposición bilateral).

    Figura principal: 3×4 paneles, un sector por panel.
    Útil para documentar la heterogeneidad estructural del NBFI
    y los cambios de composición en los episodios de estrés.
    """
    print("  Fig 2C: Composición de activos por sector (desde red)...")

    df = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df["t"] = df["periodo"].apply(tp)

    COLORES_INSTR = {
        "Prestamos":    "#2980B9",
        "Titulos deuda":"#E74C3C",
        "Depositos":    "#27AE60",
        "Cuotas FNM":   "#E67E22",
        "Cuotas FMM":   "#8E44AD",
    }
    ORDEN_INSTR = [
        "Prestamos","Titulos deuda","Depositos","Cuotas FNM","Cuotas FMM"
    ]
    LABELS_INSTR = {
        "Prestamos":    "Préstamos",
        "Titulos deuda":"Títulos de deuda",
        "Depositos":    "Depósitos",
        "Cuotas FNM":   "Cuotas FNM",
        "Cuotas FMM":   "Cuotas FMM",
    }

    SECTORES_ORDEN = [
        ("S121","Banco Central"),
        ("S122","Bancos"),
        ("S123","FMM"),
        ("S124","FNM"),
        ("S125_S126","Otros interm."),
        ("S128","Seguros"),
        ("S129","Fondos de Pensiones"),
        ("S13","Gobierno"),
        ("S11","Empresas NF"),
        ("S14_S15","Hogares"),
        ("S2","Resto del mundo"),
    ]

    # Composición porcentual por sector × instrumento × período
    comp = df.groupby(
        ["periodo","t","sector_activo_codigo","instrumento"]
    )["valor"].sum().reset_index()
    total_sector = comp.groupby(
        ["periodo","t","sector_activo_codigo"]
    )["valor"].sum().reset_index().rename(columns={"valor":"total"})
    comp = comp.merge(total_sector, on=["periodo","t","sector_activo_codigo"])
    comp["pct"] = comp["valor"] / comp["total"] * 100
    comp = comp[comp["total"] > 0]

    # Funciones auxiliares locales para paneles pequeños
    def vl_small(ax):
        for p in EPISODIOS_LINEAS.values():
            ax.axvline(tp(p), color="gray", ls=":", lw=0.9, alpha=0.7)
        t_ini, t_fin = tp("2020T3"), tp("2021T2")
        ax.axvspan(t_ini, t_fin, color="gray", alpha=0.12, zorder=0)

    def vl_small(ax):
        COLOR_EP = "#C0392B"  # rojo oscuro — solo para esta figura
        for lbl, p in EPISODIOS_LINEAS.items():
            ax.axvline(tp(p), color=COLOR_EP, ls="--", lw=1.1, alpha=1)
            ax.text(tp(p) + 0.1, 0.92, lbl,
                    fontsize=5, color=COLOR_EP,
                    va="top", ha="left",
                    transform=ax.get_xaxis_transform(),
                    linespacing=1.1)
        t_ini, t_fin = tp("2020T3"), tp("2021T2")
        ax.axvspan(t_ini, t_fin, color=COLOR_EP, alpha=0.10, zorder=3)
        ax.text((t_ini + t_fin) / 2, 0.02,
                "Retiros\nAFP",
                fontsize=5, color=COLOR_EP,
                ha="center", va="bottom",
                transform=ax.get_xaxis_transform(),
                linespacing=1.1)
        
    def xax_small(ax):
        ax.set_xlim(2002.5, 2025.8)
        ax.set_xticks([2004, 2008, 2012,2016,2020,2024])
        ax.set_xticklabels(["2004","2008","2012","2016","2020","2024"], fontsize=6)

    # Ajuste temporal de rcParams para paneles pequeños
    orig_fontsize   = plt.rcParams["font.size"]
    orig_titlesize  = plt.rcParams["axes.titlesize"]
    orig_labelsize  = plt.rcParams["axes.labelsize"]
    plt.rcParams.update({
        "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 7
    })

    nrows, ncols = 3, 4
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 9), sharey=False)
    fig.suptitle(
        "Composición de activos de intermediación por sector "
        "institucional, 2003–2025\n"
        "(% del activo total del sector en la red who-to-whom)",
        fontsize=10, y=1.01
    )

    for idx, (cod, nombre) in enumerate(SECTORES_ORDEN):
        row, col = divmod(idx, ncols)
        ax = axes[row][col]

        sub = comp[comp["sector_activo_codigo"] == cod].copy()
        if sub.empty or sub["total"].sum() == 0:
            ax.set_visible(False)
            continue

        piv = sub.pivot_table(
            index="t", columns="instrumento",
            values="pct", aggfunc="sum"
        ).reindex(columns=ORDEN_INSTR).fillna(0)

        fechas = piv.index.values
        bottom = np.zeros(len(fechas))

        for instr in ORDEN_INSTR:
            if instr not in piv.columns: continue
            vals = piv[instr].values
            ax.fill_between(fechas, bottom, bottom + vals,
                            color=COLORES_INSTR[instr],
                            alpha=0.82, label=LABELS_INSTR[instr])
            bottom += vals

        vl_small(ax)
        xax_small(ax)
        ax.set_ylim(0, 105)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.set_title(f"{cod} — {nombre}", pad=4)
        if col == 0:
            ax.set_ylabel("% activo en red", fontsize=7)

    # Celda de leyenda (posición [2][3])
    axes[2][3].set_visible(False)
    handles = [
        plt.Rectangle((0,0), 1, 1,
                       color=COLORES_INSTR[i], alpha=0.82)
        for i in ORDEN_INSTR
    ]
    labels_leg = [LABELS_INSTR[i] for i in ORDEN_INSTR]
    fig.legend(handles, labels_leg,
               loc="lower right",
               bbox_to_anchor=(0.98, 0.05),
               fontsize=9, frameon=True,
               framealpha=0.9, edgecolor="silver",
               title="Instrumento", title_fontsize=9)

    fig.text(
        0.5, -0.01,
        "Líneas punteadas: GFC 2008T3 y Estallido 2019T4. "
        "Banda gris: Retiros AFP 2020T3–2021T2.",
        ha="center", fontsize=7.5, color="dimgray", style="italic"
    )

    # Restaurar rcParams originales
    plt.rcParams.update({
        "font.size": orig_fontsize,
        "axes.titlesize": orig_titlesize,
        "axes.labelsize": orig_labelsize,
    })

    save_fig(fig, "fig2c_composicion_instrumentos")

# ============================================================
# FIG 2C SIMPLE: COMPOSICIÓN DE ACTIVOS POR SECTOR
#                (versión defensa, 6 sectores)
# ============================================================

def fig2c_simple_composicion_instrumentos():
    """
    Versión simplificada de fig2c_composicion_instrumentos para
    la presentación de defensa. Misma lógica (% del activo del
    sector en la red), misma paleta, pero acotada a 6 sectores
    clave de la narrativa: FP, Bancos, Banco Central, Seguros,
    FMM y FNM.

    Layout 2x3 en lugar de 3x4. El resto es idéntico a fig2c.
    """
    print("  Fig 2C simple: Composición por sector (6 sectores, defensa)...")

    df = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df["t"] = df["periodo"].apply(tp)

    COLORES_INSTR = {
        "Prestamos":    "#2980B9",
        "Titulos deuda":"#E74C3C",
        "Depositos":    "#27AE60",
        "Cuotas FNM":   "#E67E22",
        "Cuotas FMM":   "#8E44AD",
    }
    ORDEN_INSTR = [
        "Prestamos","Titulos deuda","Depositos","Cuotas FNM","Cuotas FMM"
    ]
    LABELS_INSTR = {
        "Prestamos":    "Préstamos",
        "Titulos deuda":"Títulos de deuda",
        "Depositos":    "Depósitos",
        "Cuotas FNM":   "Cuotas FNM",
        "Cuotas FMM":   "Cuotas FMM",
    }

    # 6 sectores clave de la narrativa de la defensa
    SECTORES_ORDEN = [
        ("S129","Fondos de Pensiones"),
        ("S122","Bancos"),
        ("S121","Banco Central"),
        ("S128","Seguros"),
        ("S123","FMM"),
        ("S124","FNM"),
    ]

    # Composición porcentual por sector x instrumento x período
    comp = df.groupby(
        ["periodo","t","sector_activo_codigo","instrumento"]
    )["valor"].sum().reset_index()
    total_sector = comp.groupby(
        ["periodo","t","sector_activo_codigo"]
    )["valor"].sum().reset_index().rename(columns={"valor":"total"})
    comp = comp.merge(total_sector, on=["periodo","t","sector_activo_codigo"])
    comp["pct"] = comp["valor"] / comp["total"] * 100
    comp = comp[comp["total"] > 0]

    # Funciones auxiliares locales (idénticas a fig2c)
    def vl_small(ax):
        COLOR_EP = "#C0392B"
        for lbl, p in EPISODIOS_LINEAS.items():
            ax.axvline(tp(p), color=COLOR_EP, ls="--", lw=1.1, alpha=1)
            ax.text(tp(p) + 0.1, 0.92, lbl,
                    fontsize=5, color=COLOR_EP,
                    va="top", ha="left",
                    transform=ax.get_xaxis_transform(),
                    linespacing=1.1)
        t_ini, t_fin = tp("2020T3"), tp("2021T2")
        ax.axvspan(t_ini, t_fin, color=COLOR_EP, alpha=0.10, zorder=3)
        ax.text((t_ini + t_fin) / 2, 0.02,
                "Retiros\nAFP",
                fontsize=5, color=COLOR_EP,
                ha="center", va="bottom",
                transform=ax.get_xaxis_transform(),
                linespacing=1.1)

    def xax_small(ax):
        ax.set_xlim(2002.5, 2025.8)
        ax.set_xticks([2004, 2008, 2012, 2016, 2020, 2024])
        ax.set_xticklabels(["2004","2008","2012","2016","2020","2024"], fontsize=6)

    # Ajuste temporal de rcParams para paneles pequeños
    orig_fontsize  = plt.rcParams["font.size"]
    orig_titlesize = plt.rcParams["axes.titlesize"]
    orig_labelsize = plt.rcParams["axes.labelsize"]
    plt.rcParams.update({
        "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 7
    })

    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 6.5), sharey=False)
    fig.suptitle(
        "Composición de activos de intermediación por sector institucional, 2003-2025\n"
        "(% del activo total del sector en la red who-to-whom)",
        fontsize=10, y=1.01
    )

    for idx, (cod, nombre) in enumerate(SECTORES_ORDEN):
        row, col = divmod(idx, ncols)
        ax = axes[row][col]

        sub = comp[comp["sector_activo_codigo"] == cod].copy()
        if sub.empty or sub["total"].sum() == 0:
            ax.set_visible(False)
            continue

        piv = sub.pivot_table(
            index="t", columns="instrumento",
            values="pct", aggfunc="sum"
        ).reindex(columns=ORDEN_INSTR).fillna(0)

        fechas = piv.index.values
        bottom = np.zeros(len(fechas))

        for instr in ORDEN_INSTR:
            if instr not in piv.columns: continue
            vals = piv[instr].values
            ax.fill_between(fechas, bottom, bottom + vals,
                            color=COLORES_INSTR[instr],
                            alpha=0.82, label=LABELS_INSTR[instr])
            bottom += vals

        vl_small(ax)
        xax_small(ax)
        ax.set_ylim(0, 105)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.set_title(f"{cod} — {nombre}", pad=4)
        if col == 0:
            ax.set_ylabel("% activo en red", fontsize=7)

    # Leyenda única abajo
    handles = [
        plt.Rectangle((0,0), 1, 1,
                       color=COLORES_INSTR[i], alpha=0.82)
        for i in ORDEN_INSTR
    ]
    labels_leg = [LABELS_INSTR[i] for i in ORDEN_INSTR]
    fig.legend(handles, labels_leg,
               loc="lower center",
               bbox_to_anchor=(0.5, -0.07),
               ncol=5, fontsize=9, frameon=True,
               framealpha=0.9, edgecolor="silver",
               title="Instrumento", title_fontsize=9)

    fig.text(
        0.5, -0.09,
        "Líneas punteadas: GFC 2008T3 y Estallido 2019T4. "
        "Banda gris: Retiros AFP 2020T3-2021T2.",
        ha="center", fontsize=7.5, color="dimgray", style="italic"
    )

    # Restaurar rcParams
    plt.rcParams.update({
        "font.size": orig_fontsize,
        "axes.titlesize": orig_titlesize,
        "axes.labelsize": orig_labelsize,
    })

    save_fig(fig, "fig2c_simple_composicion_instrumentos")
# ============================================================
# FIG 2D: COMPOSICIÓN DE PASIVOS POR SECTOR (DESDE RED)
# ============================================================

def fig2d_composicion_pasivos():
    """
    Composición porcentual de pasivos de intermediación por sector
    institucional, construida desde panel_red_saldos.csv.

    Cada panel muestra qué fracción del pasivo total del sector
    en la red está en cada instrumento (préstamos, títulos de
    deuda, depósitos, cuotas FNM, cuotas FMM).

    Complementa fig2c (activos): mientras fig2c muestra en qué
    invierte cada sector, fig2d muestra cómo se financia.

    Nota: FP (S129) tiene pasivos = 0 en la red por construcción.
    Sus pasivos son cuotas de pensiones registradas fuera de la red.
    Hogares (S14_S15) tiene pasivos solo en préstamos.
    Empresas NF (S11) tiene pasivos en préstamos y títulos.
    """
    print("  Fig 2D: Composición de pasivos por sector (desde red)...")

    df = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df["t"] = df["periodo"].apply(tp)

    COLORES_INSTR = {
        "Prestamos":    "#2980B9",
        "Titulos deuda":"#E74C3C",
        "Depositos":    "#27AE60",
        "Cuotas FNM":   "#E67E22",
        "Cuotas FMM":   "#8E44AD",
    }
    ORDEN_INSTR = [
        "Prestamos","Titulos deuda","Depositos","Cuotas FNM","Cuotas FMM"
    ]
    LABELS_INSTR = {
        "Prestamos":    "Préstamos",
        "Titulos deuda":"Títulos de deuda",
        "Depositos":    "Depósitos",
        "Cuotas FNM":   "Cuotas FNM",
        "Cuotas FMM":   "Cuotas FMM",
    }

    SECTORES_ORDEN = [
        ("S121","Banco Central"),
        ("S122","Bancos"),
        ("S123","FMM"),
        ("S124","FNM"),
        ("S125_S126","Otros interm."),
        ("S128","Seguros"),
        ("S129","Fondos de Pensiones"),
        ("S13","Gobierno"),
        ("S11","Empresas NF"),
        ("S14_S15","Hogares"),
        ("S2","Resto del mundo"),
    ]

    # Composición porcentual por sector PASIVO × instrumento × período
    comp = df.groupby(
        ["periodo","t","sector_pasivo_codigo","instrumento"]
    )["valor"].sum().reset_index()
    total_sector = comp.groupby(
        ["periodo","t","sector_pasivo_codigo"]
    )["valor"].sum().reset_index().rename(columns={"valor":"total"})
    comp = comp.merge(total_sector, on=["periodo","t","sector_pasivo_codigo"])
    comp["pct"] = comp["valor"] / comp["total"] * 100
    comp = comp[comp["total"] > 0]

       # Funciones auxiliares locales para paneles pequeños
    def vl_small(ax):
        for p in EPISODIOS_LINEAS.values():
            ax.axvline(tp(p), color="gray", ls=":", lw=0.9, alpha=0.7)
        t_ini, t_fin = tp("2020T3"), tp("2021T2")
        ax.axvspan(t_ini, t_fin, color="gray", alpha=0.12, zorder=0)

    def vl_small(ax):
        COLOR_EP = "#C0392B"  # rojo oscuro — solo para esta figura
        for lbl, p in EPISODIOS_LINEAS.items():
            ax.axvline(tp(p), color=COLOR_EP, ls="--", lw=1.1, alpha=1)
            ax.text(tp(p) + 0.1, 0.92, lbl,
                    fontsize=5, color=COLOR_EP,
                    va="top", ha="left",
                    transform=ax.get_xaxis_transform(),
                    linespacing=1.1)
        t_ini, t_fin = tp("2020T3"), tp("2021T2")
        ax.axvspan(t_ini, t_fin, color=COLOR_EP, alpha=0.10, zorder=3)
        ax.text((t_ini + t_fin) / 2, 0.02,
                "Retiros\nAFP",
                fontsize=5, color=COLOR_EP,
                ha="center", va="bottom",
                transform=ax.get_xaxis_transform(),
                linespacing=1.1)
        
    def xax_small(ax):
        ax.set_xlim(2002.5, 2025.8)
        ax.set_xticks([2004, 2008, 2012,2016,2020,2024])
        ax.set_xticklabels(["2004","2008","2012","2016","2020","2024"], fontsize=6)

    orig_fontsize  = plt.rcParams["font.size"]
    orig_titlesize = plt.rcParams["axes.titlesize"]
    orig_labelsize = plt.rcParams["axes.labelsize"]
    plt.rcParams.update({
        "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 7
    })

    nrows, ncols = 3, 4
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 9), sharey=False)
    fig.suptitle(
        "Composición de pasivos de intermediación por sector "
        "institucional, 2003–2025\n"
        "(% del pasivo total del sector en la red who-to-whom)",
        fontsize=10, y=1.01
    )

    for idx, (cod, nombre) in enumerate(SECTORES_ORDEN):
        row, col = divmod(idx, ncols)
        ax = axes[row][col]

        sub = comp[comp["sector_pasivo_codigo"] == cod].copy()

        # FP tiene pasivos = 0 → mostrar panel vacío con nota
        if sub.empty or sub["total"].sum() == 0:
            ax.set_xlim(2002.5, 2025.8)
            ax.set_ylim(0, 105)
            ax.set_title(f"{cod} — {nombre}", pad=4)
            ax.text(0.5, 0.5,
                    "Sin pasivos\nen la red",
                    ha="center", va="center",
                    fontsize=8, color="gray",
                    transform=ax.transAxes)
            xax_small(ax)
            if col == 0:
                ax.set_ylabel("% pasivo en red", fontsize=7)
            continue

        piv = sub.pivot_table(
            index="t", columns="instrumento",
            values="pct", aggfunc="sum"
        ).reindex(columns=ORDEN_INSTR).fillna(0)

        fechas = piv.index.values
        bottom = np.zeros(len(fechas))

        for instr in ORDEN_INSTR:
            if instr not in piv.columns: continue
            vals = piv[instr].values
            ax.fill_between(fechas, bottom, bottom + vals,
                            color=COLORES_INSTR[instr],
                            alpha=0.82, label=LABELS_INSTR[instr])
            bottom += vals

        vl_small(ax)
        xax_small(ax)
        ax.set_ylim(0, 105)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_yticks([0, 25, 50, 75, 100])
        ax.set_title(f"{cod} — {nombre}", pad=4)
        if col == 0:
            ax.set_ylabel("% pasivo en red", fontsize=7)

    # Celda de leyenda
    axes[2][3].set_visible(False)
    handles = [
        plt.Rectangle((0,0), 1, 1,
                       color=COLORES_INSTR[i], alpha=0.82)
        for i in ORDEN_INSTR
    ]
    labels_leg = [LABELS_INSTR[i] for i in ORDEN_INSTR]
    fig.legend(handles, labels_leg,
               loc="lower right",
               bbox_to_anchor=(0.98, 0.05),
               fontsize=9, frameon=True,
               framealpha=0.9, edgecolor="silver",
               title="Instrumento", title_fontsize=9)

    fig.text(
        0.5, -0.01,
        "Líneas punteadas: GFC 2008T3 y Estallido 2019T4. "
        "Banda gris: Retiros AFP 2020T3–2021T2. "
        "AFP (S129): pasivos = 0 en la red por construcción "
        "(sus pasivos son cuotas previsionales fuera de la red).",
        ha="center", fontsize=7.5, color="dimgray", style="italic"
    )

    plt.rcParams.update({
        "font.size": orig_fontsize,
        "axes.titlesize": orig_titlesize,
        "axes.labelsize": orig_labelsize,
    })

    save_fig(fig, "fig2d_composicion_pasivos")
    
# ============================================================
# FIG 3: Exposición bilateral NBFI -> Bancos vs Bancos -> NBFI (asimetría)
# ============================================================

def fig3_exposicion_bilateral_asimetria():
    """
    Asimetría central de la tesis: NBFI concentra ~32% de su activo
    en instrumentos bancarios; bancos destinan solo ~5% en NBFI.
    Panel A: Ratio de exposición (% activo total en la red)
    Panel B: Niveles absolutos (MMCL)
    """
    print("  Fig 3: Asimetría exposición bilateral NBFI↔Bancos...")

    df_agg = pd.read_csv(DATA_DIR / "panel_exp_nbfi_bancos_agregado.csv")
    df_bn  = pd.read_csv(DATA_DIR / "panel_exp_bancos_nbfi.csv")
    df_agg["t"] = df_agg["periodo"].apply(tp)
    df_bn["t"]  = df_bn["periodo"].apply(tp)

    agg = df_agg.sort_values("t")
    bn  = df_bn.sort_values("t")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        "Exposición bilateral entre NBFIs y Bancos, 2003–2025\n"
        "(denominador: activo total del sector en la red who-to-whom, "
        "excluye dinero legal)",
        fontsize=11
    )

    # ── Panel A: ratios ───────────────────────────────────────────────
    ax = axes[0]
    ax.plot(agg["t"], agg["ratio_exp_nbfi_bancos_agg"] * 100,
            color=COLORS["NBFI"], lw=2.0, label="NBFI → Bancos (agregado)")
    ax.plot(bn["t"],  bn["ratio_exp_bancos_nbfi"] * 100,
            color=COLORS["S122"], lw=2.0, ls="--", label="Bancos → NBFI")

    # Líneas de media
    mean_nb = agg["ratio_exp_nbfi_bancos_agg"].mean() * 100
    mean_bn = bn["ratio_exp_bancos_nbfi"].mean() * 100
    ax.axhline(mean_nb, color=COLORS["NBFI"], ls=":", lw=0.8, alpha=0.5)
    ax.axhline(mean_bn, color=COLORS["S122"], ls=":", lw=0.8, alpha=0.5)
    ax.text(2025.5, mean_nb + 0.5, f"{mean_nb:.1f}%",
            fontsize=7, color=COLORS["NBFI"], va="bottom", ha="right")
    ax.text(2025.5, mean_bn + 0.5, f"{mean_bn:.1f}%",
            fontsize=7, color=COLORS["S122"], va="bottom", ha="right")

    ax.set_title("A. Ratio de exposición (% activos totales del sector)")
    ax.set_ylabel("% del activo total del sector")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_yticks(np.arange(0, 55, 10))
    xax(ax); vl(ax)
    legend_below(ax, ncol=2)

    # ── Panel B: niveles ──────────────────────────────────────────────
    ax2 = axes[1]
    # exp_en_bancos = ratio × activo_nbfi_red_total
    agg["exp_en_bancos_calc"] = (
        agg["ratio_exp_nbfi_bancos_agg"] * agg["activo_nbfi_red_total"]
    )
    ax2.plot(agg["t"], agg["exp_en_bancos_calc"] / 1000,
             color=COLORS["NBFI"], lw=2.0, label="NBFI → Bancos")
    ax2.plot(bn["t"],  bn["exp_bancos_en_nbfi"] / 1000,
             color=COLORS["S122"], lw=2.0, ls="--", label="Bancos → NBFI")
    
    ax2.set_title("B. Niveles (miles de MMCL)")
    ax2.set_ylabel("Miles de MMCL (billones escala larga)")
    xax(ax2); vl(ax2)
    legend_below(ax2, ncol=2)

    save_fig(fig, "fig3_exposicion_asimetria")

# ============================================================
# FIG 4: Exposición por sector NBFI → Bancos (heterogeneidad)
# ============================================================

def fig4_exposicion_sector_nbfi():
    """
    Heterogeneidad interna del NBFI en su exposición hacia bancos.
    Panel A: Ratio por sector (% activo en red) — líneas
    Panel B: Composición apilada del monto total NBFI en bancos
    """
    print("  Fig 4: Exposición por sector NBFI hacia bancos...")

    df = pd.read_csv(DATA_DIR / "panel_exp_nbfi_bancos_sector.csv")
    df["t"] = df["periodo"].apply(tp)

    # Recalcular monto absoluto desde ratio × denominador
    # (el archivo no guarda el monto directo)
    df["exp_en_bancos_calc"] = (
        df["ratio_exp_nbfi_bancos"] * df["activo_red_nbfi_sector"]
    )

    SECTORES = [
        ("S129",      "AFP",         COLORS["S129"],      "-",  2.0),
        ("S123",      "FMM",         COLORS["S123"],      "-",  2.0),
        ("S128",      "Seguros",     COLORS["S128"],      "--", 1.6),
        ("S124",      "FNM",         COLORS["S124"],      "--", 1.6),
        ("S125_S126", "Otros interm.", COLORS["S125_S126"], ":", 1.4),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        "Exposición de cada sector NBFI en instrumentos bancarios, 2003–2025\n"
        "(denominador: activo total del sector en la red who-to-whom, "
        "excluye dinero legal)",
        fontsize=11
    )

    # ── Panel A: ratios por sector ────────────────────────────────────
    ax = axes[0]
    for cod, lbl, col, ls, lw in SECTORES:
        sub = df[df["sector_codigo"] == cod].sort_values("t")
        ax.plot(sub["t"], sub["ratio_exp_nbfi_bancos"] * 100,
                color=col, lw=lw, ls=ls, label=lbl)

    ax.set_title("A. Ratio de exposición por sector (% activos totales del sector)")
    ax.set_ylabel("% del activo total del sector")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_yticks(np.arange(0, 110, 20))
    xax(ax); vl(ax)
    legend_below(ax, ncol=3)

    # ── Panel B: área apilada (monto recalculado) ─────────────────────
    ax2 = axes[1]

    orden_apilado = ["S125_S126", "S124", "S128", "S123", "S129"]
    labels_apilado = {
        "S125_S126": "Otros interm.",
        "S124":      "FNM",
        "S128":      "Seguros",
        "S123":      "FMM",
        "S129":      "AFP",
    }

    fechas = sorted(df["t"].unique())
    bottom = np.zeros(len(fechas))

    for cod in orden_apilado:
        sub = df[df["sector_codigo"] == cod].sort_values("t")
        sub = sub.set_index("t").reindex(fechas).fillna(0)
        vals = sub["exp_en_bancos_calc"].values / 1000
        ax2.fill_between(fechas, bottom, bottom + vals,
                         alpha=0.75, color=COLORS[cod],
                         label=labels_apilado[cod])
        bottom += vals

    ax2.set_title("B. Monto NBFI en bancos por sector (billones CLP)")
    ax2.set_ylabel("Miles de MMCL (billones escala larga)")
    xax(ax2); vl(ax2)
    legend_below(ax2, ncol=3)

    save_fig(fig, "fig4_exposicion_sector_nbfi")
# ============================================================
# FIG 3.2 IS_BILATERAL: ÍNDICE DE DEPENDENCIA DE FONDEO BILATERAL
# ============================================================

def fig_is_bilateral_fondeo():
    """
    IS bilateral de dependencia de fondeo en ambas direcciones.

    IS^Activo_{i→j,t} = e_{ij,t} / A^j_{red,t}
    Responde: ¿qué fracción del fondeo del sector j proviene de i?

    Panel A: NBFI→Bancos — desagregado por sector NBFI
             ¿qué fracción del fondeo bancario proviene de cada NBFI?
    Panel B: Bancos→NBFI — desagregado por sector NBFI
             ¿qué fracción del fondeo de cada NBFI proviene de bancos?

    Nota: S125_S126 excluido del gráfico por denominador pequeño
    (su activo en la red es mínimo al excluir acciones).
    """
    print("  Fig IS Bilateral: Dependencia de fondeo NBFI↔Bancos (desagregado)...")

    df_red = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df_red["t"] = df_red["periodo"].apply(tp)

    NBFI_PLOT = ["S123","S124","S128","S129"]  # excluye S125_S126
    BANCARIO  = ["S122"]

    SECTORES_NBFI = [
        ("S129", "AFP",     COLORS["S129"], "-",  2.0),
        ("S128", "Seguros", COLORS["S128"], "--", 1.6),
        ("S124", "FNM",     COLORS["S124"], "--", 1.6),
        ("S123", "FMM",     COLORS["S123"], ":",  1.4),
    ]

    # Pasivo total de bancos en la red
    pas_bancos = df_red[
        df_red["sector_pasivo_codigo"].isin(BANCARIO)
    ].groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"P_bancos_red"})

    # Pasivo total de cada NBFI en la red
    pas_nbfi = df_red[
        df_red["sector_pasivo_codigo"].isin(NBFI)
    ].groupby(["periodo","t","sector_pasivo_codigo"])["valor"].sum().reset_index().rename(
        columns={"sector_pasivo_codigo":"sector_codigo","valor":"P_nbfi_red"})

    # ── Numerador: NBFI→Bancos por sector ─────────────────
    exp_nb = df_red[
        (df_red["sector_activo_codigo"].isin(NBFI_PLOT)) &
        (df_red["sector_pasivo_codigo"].isin(BANCARIO))
    ].groupby(["periodo","t","sector_activo_codigo"])["valor"].sum().reset_index().rename(
        columns={"sector_activo_codigo":"sector_codigo","valor":"e_i_bancos"})

    is_nb = exp_nb.merge(pas_bancos, on=["periodo","t"])
    is_nb["IS"] = is_nb["e_i_bancos"] / is_nb["P_bancos_red"]

    # NBFI agregado→Bancos
    nb_agg_num = exp_nb.groupby(["periodo","t"])["e_i_bancos"].sum().reset_index()
    nb_agg = nb_agg_num.merge(pas_bancos, on=["periodo","t"])
    nb_agg["IS_agg"] = nb_agg["e_i_bancos"] / nb_agg["P_bancos_red"]

    # ── Numerador: Bancos→NBFI por sector ─────────────────
    exp_bn = df_red[
        (df_red["sector_activo_codigo"].isin(BANCARIO)) &
        (df_red["sector_pasivo_codigo"].isin(NBFI_PLOT))
    ].groupby(["periodo","t","sector_pasivo_codigo"])["valor"].sum().reset_index().rename(
        columns={"sector_pasivo_codigo":"sector_codigo","valor":"e_bancos_i"})

    is_bn = exp_bn.merge(pas_nbfi, on=["periodo","t","sector_codigo"])
    is_bn["IS"] = is_bn["e_bancos_i"] / is_bn["P_nbfi_red"]

    # Bancos→NBFI agregado
    pas_nbfi_agg = pas_nbfi.groupby(["periodo","t"])["P_nbfi_red"].sum().reset_index()
    bn_agg_num = exp_bn.groupby(["periodo","t"])["e_bancos_i"].sum().reset_index()
    bn_agg = bn_agg_num.merge(pas_nbfi_agg, on=["periodo","t"])
    bn_agg["IS_agg"] = bn_agg["e_bancos_i"] / bn_agg["P_nbfi_red"]

    # ── Figura ────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        r"Índice de Dependencia de Fondeo Bilateral ($IS^{Pasivo}_{i \rightarrow j}$)"
        ", 2003–2025\n"
        r"$IS^{Pasivo}_{i \rightarrow j,t} = e_{ij,t} / P^j_{red,t}$ "
        "| denominador: pasivo total del sector deudor en la red",
        fontsize=11
    )

    # Panel A: NBFI→Bancos
    ax = axes[0]
    for cod, lbl, col, ls, lw in SECTORES_NBFI:
        sub = is_nb[is_nb["sector_codigo"]==cod].sort_values("t")
        if sub.empty: continue
        ax.plot(sub["t"], sub["IS"]*100,
                color=col, lw=lw, ls=ls, label=lbl)
    mean_a = nb_agg["IS_agg"].mean()*100
    ax.set_title(
        r"A. $IS^{Activo}_{NBFI \rightarrow Bancos}$: "
        "¿qué fracción del fondeo\nbancario proviene del NBFI?"
    )
    ax.set_ylabel("% del pasivo bancario total en la red")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylim(0, 55)
    ax.set_yticks(np.arange(0, 60, 10))
    xax(ax); vl(ax)
    legend_below(ax, ncol=3)

    # Panel B: Bancos→NBFI
    ax2 = axes[1]
    for cod, lbl, col, ls, lw in SECTORES_NBFI:
        sub = is_bn[is_bn["sector_codigo"]==cod].sort_values("t")
        if sub.empty: continue
        ax2.plot(sub["t"], sub["IS"]*100,
                 color=col, lw=lw, ls=ls, label=f"Bancos → {lbl}")
    mean_b = bn_agg["IS_agg"].mean()*100
    ax2.set_title(
        r"B. $IS^{Activo}_{Bancos \rightarrow NBFI}$: "
        "¿qué fracción del fondeo\nNBFI proviene de los bancos?"
    )
    ax2.set_ylabel("% del pasivo NBFI total en la red")
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax2.set_ylim(0, 20)
    ax2.set_yticks(np.arange(0, 25, 5))
    xax(ax2); vl(ax2)
    legend_below(ax2, ncol=3)

    save_fig(fig, "fig_is_bilateral_fondeo")

# ============================================================
# FIG 3.C: ROL DEL BANCO CENTRAL DURANTE PANDEMIA Y RETIROS
# Activos del BC en la red who-to-whom como % del PIB, área
# apilada por instrumento + descomposición del Δ 2019T4→2021T2
# por (instrumento, contraparte).
# ============================================================

def fig_bc_rol_pandemia():
    """
    Rol del Banco Central como mitigador durante pandemia y retiros.

    Panel A: activos del BC (S121) en la red who-to-whom como % del PIB
             (suma móvil 4 trim.), área apilada por instrumento.
    Panel B: descomposición del cambio de saldo 2019T4 -> 2021T2 por
             (instrumento, contraparte). Identifica FCIC (Préstamos a
             Bancos) y CC-VP (Títulos a Bancos) como destino dominante.

    Insumos:
      - panel_red_saldos.csv (5 instrumentos de intermediación)
      - build_pib_df() para denominador PIB MM4
    """
    print("  Fig 3.C: Rol del BC durante pandemia y retiros...")

    df_red = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    bc = df_red[df_red.sector_activo_codigo == "S121"].copy()

    # ---------- Panel A: stock BC por instrumento, % del PIB ----------
    ORDEN_INSTR = ["Titulos deuda", "Depositos", "Prestamos",
                   "Cuotas FMM", "Cuotas FNM"]
    LABEL_INSTR = {
        "Depositos":     "Depósitos",
        "Prestamos":     "Préstamos",
        "Titulos deuda": "Títulos de deuda",
        "Cuotas FMM":    "Cuotas FMM",
        "Cuotas FNM":    "Cuotas FNM",
    }

    piv = (bc.groupby(["periodo", "instrumento"])["valor"].sum()
             .unstack(fill_value=0)
             .reset_index())
    piv["t"] = piv["periodo"].apply(tp)

    pib_df = build_pib_df()
    piv = piv.merge(pib_df[["periodo", "pib_mm4"]], on="periodo", how="left")
    piv = piv.dropna(subset=["pib_mm4"]).sort_values("t").reset_index(drop=True)

    for instr in ORDEN_INSTR:
        if instr not in piv.columns:
            piv[instr] = 0.0
        piv[f"pct_{instr}"] = piv[instr] / piv["pib_mm4"] * 100

    # ---------- Panel B: Δ saldo 2019T4 -> 2021T2 por (instr, contraparte) ----------
    piv_ic = bc.pivot_table(
        index="periodo",
        columns=["instrumento", "sector_pasivo_nombre"],
        values="valor", aggfunc="sum").fillna(0)
    ini, fin = "2019T4", "2021T2"
    delta = ((piv_ic.loc[fin] - piv_ic.loc[ini])
             .rename("delta").reset_index())

    short_pas = {
        "Bancos y cooperativas":                                     "Bancos",
        "Resto del mundo":                                            "Resto del Mundo",
        "Gobierno general":                                           "Gobierno",
        "Empresas no financieras":                                    "Empresas NF",
        "Otros intermediarios financieros y Auxiliares financieros":  "Otros interm.",
        "Hogares":                                                    "Hogares",
        "Compañías de seguro":                                        "Seguros",
        "Fondo de pensiones":                                         "AFP",
        "Fondos del mercado monetario":                               "FMM",
        "Fondos del mercado no monetario":                            "FNM",
    }
    delta["pas_lbl"]   = delta["sector_pasivo_nombre"].map(lambda s: short_pas.get(s, s))
    delta["instr_lbl"] = delta["instrumento"].map(LABEL_INSTR)
    delta["bar_lbl"]   = delta["instr_lbl"] + " → " + delta["pas_lbl"]
    delta["color"]     = delta["instrumento"].map(COLORES_INSTR)
    # Filtrar ruido: mantener solo barras ≥ 0.5% del cambio total absoluto
    total_abs = delta["delta"].abs().sum()
    delta = (delta[delta["delta"].abs() >= 0.005 * total_abs]
                .sort_values("delta")
                .reset_index(drop=True))

    # ---------- FIGURA ----------
    fig = plt.figure(figsize=(14, 5.4))
    gs  = fig.add_gridspec(1, 2, width_ratios=[1.3, 0.8], wspace=0.45) #ESTO AJUSTO PARA QUE NO SE TRASLAPEN!
    fig.suptitle(
        "El rol del Banco Central durante la pandemia y los retiros previsionales, 2003–2025\n"
        "Activos en la red who-to-whom (Préstamos, Depósitos, Títulos de deuda, Cuotas FMM y FNM)",
        fontsize=11.5, y=1.02)

    # Panel A
    axA = fig.add_subplot(gs[0, 0])
    x  = piv["t"].values
    ys = np.vstack([piv[f"pct_{i}"].values for i in ORDEN_INSTR])
    colors = [COLORES_INSTR[i] for i in ORDEN_INSTR]
    labels = [LABEL_INSTR[i]   for i in ORDEN_INSTR]
    axA.stackplot(x, ys, colors=colors, labels=labels, alpha=0.85,
                  edgecolor="white", linewidth=0.4)
    total = ys.sum(axis=0)
    axA.plot(x, total, color="black", lw=1.0, alpha=0.7)

    axA.set_title("A. Activos del Banco Central por instrumento, % del PIB", loc="left")
    axA.set_ylabel("% del PIB (suma móvil 4 trim.)")
    axA.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    xax(axA)
    axA.set_ylim(0, float(total.max()) * 1.55)
    vl(axA)

    # Anotar pico durante pandemia/retiros
    mask_p = (x >= 2020.0) & (x <= 2022.0)
    idx_p  = int(np.argmax(total * mask_p))
    axA.annotate(
        f"Máx. pandemia/retiros:\n{total[idx_p]:.1f}% PIB ({piv['periodo'].iloc[idx_p]})",
        xy=(x[idx_p], total[idx_p]),
        xytext=(2014.5, total[idx_p] + 8.0),
        fontsize=8.5, color="black",
        arrowprops=dict(arrowstyle="->", color="black", lw=0.7,
                        connectionstyle="arc3,rad=-0.15"))
    axA.legend(loc="upper left", fontsize=8, frameon=True,
               framealpha=0.95, edgecolor="silver", ncol=1,
               bbox_to_anchor=(0.01, 0.99))

    # Panel B
    axB = fig.add_subplot(gs[0, 1])
    y_pos = np.arange(len(delta))
    axB.barh(y_pos, delta["delta"].values,
             color=delta["color"].values,
             edgecolor="white", linewidth=0.5, alpha=0.9)
    axB.axvline(0, color="black", lw=0.6)
    axB.set_yticks(y_pos)
    axB.set_yticklabels(delta["bar_lbl"].values, fontsize=8)
    axB.set_xlabel("Δ saldo 2019T4 → 2021T2  (miles de MM CLP)")
    axB.set_title("B. ¿A dónde fue el incremento? Descomposición\n"
                  "por instrumento y contraparte",
                  loc="left", fontsize=10.5)
    axB.grid(axis="y", visible=False)
    axB.grid(axis="x", alpha=0.3, linestyle="--")

    xmax_b = float(delta["delta"].max()) * 1.18
    xmin_b = min(float(delta["delta"].min()) * 4.5, -6000)
    axB.set_xlim(xmin_b, xmax_b)

    def _fmt_es(v):
        return f"{v:,.0f}".replace(",", ".")

    for i, v in enumerate(delta["delta"].values):
        if v >= 0:
            axB.text(v + xmax_b * 0.012, i, _fmt_es(v),
                     va="center", ha="left",  fontsize=8, color="black")
        else:
            axB.text(v - xmax_b * 0.012, i, _fmt_es(v),
                     va="center", ha="right", fontsize=8, color="black")

    axB.text(0.98, 0.06,
             "FCIC → Préstamos a Bancos\nCC-VP → Compra de bonos bancarios",
             transform=axB.transAxes, fontsize=7.5, ha="right", va="bottom",
             color="dimgray",
             bbox=dict(boxstyle="round,pad=0.35",
                       facecolor="white", edgecolor="silver", alpha=0.85))

    #fig.text(0.01, -0.04,
        #"Nota: El Banco Central se considera como sector acreedor (S121) en la matriz who-to-whom de las CNSI. "
        #"Se incluyen los cinco instrumentos de intermediación financiera: Préstamos, Depósitos, Títulos de deuda, "
        #"Cuotas FMM y Cuotas FNM. El denominador es el PIB nominal anualizado (suma móvil de 4 trimestres). "
        #"El Panel B contrasta los saldos bilaterales de 2019T4 (trimestre previo al inicio de la pandemia) "
        #"con 2021T2 (trimestre del tercer retiro). Fuente: Elaboración propia con datos de CNSI del Banco Central de Chile.",
        #fontsize=7.2, color="dimgray", ha="left", wrap=True)

    save_fig(fig, "fig_3C_bc_rol_pandemia")


# Llamada
fig_bc_rol_pandemia()

# ============================================================
# FIG 3.3 IS_BILATERAL_AGG: VERSIÓN SOLO CON NBFI AGREGADO
# ============================================================

def fig_is_bilateral_fondeo_agregado():
    """
    Versión simplificada: solo muestra NBFI como conjunto vs Bancos.

    IS^Activo_{i→j,t} = e_{ij,t} / P^j_{red,t}
    Denominador CORREGIDO: pasivo total del sector deudor j en la red
    (= todo lo que el SF tiene colocado en j como deudor)
    NO el activo del acreedor i.

    Interpretación:
    IS_nb: de todo el fondeo que los bancos reciben del SF,
           ¿qué fracción proviene del NBFI?
    IS_bn: de todo el fondeo que el NBFI recibe del SF,
           ¿qué fracción proviene de los bancos?

    Panel A: ratios IS en ambas direcciones (asimetría)
    Panel B: niveles absolutos (billones CLP)
    """
    print("  Fig IS Bilateral Agregado: NBFI↔Bancos (agregado)...")

    df_red = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df_red["t"] = df_red["periodo"].apply(tp)

    NBFI     = ["S123","S124","S125_S126","S128","S129"]
    BANCARIO = ["S122"]

    # ── Denominadores: pasivo total del sector deudor ─────
    # Pasivo de bancos = todo lo que el SF tiene colocado en bancos
    pas_bancos = df_red[
        df_red["sector_pasivo_codigo"].isin(BANCARIO)
    ].groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"P_bancos"})

    # Pasivo de NBFI = todo lo que el SF tiene colocado en NBFI
    pas_nbfi = df_red[
        df_red["sector_pasivo_codigo"].isin(NBFI)
    ].groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"P_nbfi"})

    # ── Numeradores: exposiciones bilaterales ─────────────
    e_nb = df_red[
        (df_red["sector_activo_codigo"].isin(NBFI)) &
        (df_red["sector_pasivo_codigo"].isin(BANCARIO))
    ].groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"e_nbfi_bancos"})

    e_bn = df_red[
        (df_red["sector_activo_codigo"].isin(BANCARIO)) &
        (df_red["sector_pasivo_codigo"].isin(NBFI))
    ].groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"e_bancos_nbfi"})

    # ── IS bilaterales con denominador correcto ───────────
    df = pas_bancos.merge(pas_nbfi, on=["periodo","t"])
    df = df.merge(e_nb, on=["periodo","t"])
    df = df.merge(e_bn, on=["periodo","t"])

    # NBFI→Bancos: fracción del fondeo bancario que viene del NBFI
    df["IS_nb"] = df["e_nbfi_bancos"] / df["P_bancos"]

    # Bancos→NBFI: fracción del fondeo NBFI que viene de los bancos
    df["IS_bn"] = df["e_bancos_nbfi"] / df["P_nbfi"]

    df = df.sort_values("t")

    fig, axes = plt.subplots(2, 1, figsize=(10, 13))
    fig.suptitle(
        r"Índice de Dependencia de Fondeo Bilateral ($IS^{Pasivo}_{i \rightarrow j}$)"
        " — NBFI y Bancos agregados, 2003–2025\n"
        r"$IS^{Pasivo}_{i \rightarrow j,t} = e_{ij,t} / P^j_{red,t}$ "
        "| denominador: pasivo total del sector deudor en la red",
        fontsize=11
    )

    # ── Panel A: ratios ───────────────────────────────────
    ax = axes[0]
    ax.plot(df["t"], df["IS_nb"]*100,
            color=COLORS["S129"], lw=2.2, ls="-",
            label="NBFI → Bancos")
    ax.plot(df["t"], df["IS_bn"]*100,
            color=COLORS["S122"], lw=2.2, ls="--",
            label="Bancos → NBFI")

    mean_nb = df["IS_nb"].mean()*100
    mean_bn = df["IS_bn"].mean()*100
    ax.axhline(mean_nb, color=COLORS["S129"], ls=":", lw=0.8, alpha=0.5)
    ax.axhline(mean_bn, color=COLORS["S122"], ls=":", lw=0.8, alpha=0.5)
    ax.text(2025.5, mean_nb+0.5, f"{mean_nb:.1f}%",
            fontsize=7, color=COLORS["S129"], va="bottom", ha="right")
    ax.text(2025.5, mean_bn+0.1, f"{mean_bn:.1f}%",
            fontsize=7, color=COLORS["S122"], va="bottom", ha="right")

    ax.set_title(
        "A. Dependencia de fondeo bilateral (%)\n"
        r"$IS^{Pasivo}_{i \rightarrow j} = e_{ij} / P^j_{red}$"
    )
    ax.set_ylabel("% del pasivo total del sector deudor en la red")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylim(0, 55)
    ax.set_yticks(np.arange(0, 60, 10))
    xax(ax); vl(ax)
    legend_below(ax, ncol=2)

    # ── Panel B: niveles ──────────────────────────────────
    ax2 = axes[1]
    ax2.plot(df["t"], df["e_nbfi_bancos"]/1000,
             color=COLORS["S129"], lw=2.2, ls="-",
             label="NBFI → Bancos")
    ax2.plot(df["t"], df["e_bancos_nbfi"]/1000,
             color=COLORS["S122"], lw=2.2, ls="--",
             label="Bancos → NBFI")

    ax2.set_title("B. Niveles de exposición bilateral (billones CLP)")
    ax2.set_ylabel("Miles de MMCL (billones CLP)")
    xax(ax2); vl(ax2)
    legend_below(ax2, ncol=2)

    save_fig(fig, "fig_is_bilateral_fondeo_agregado_2x1")
# ============================================================
# FIG IS BILATERAL FONDEO — SOLO PANEL A (defensa)
# ============================================================

def fig_is_bilateral_fondeo_solo_panelA():
    """
    Versión simplificada de fig_is_bilateral_fondeo_agregado para
    la presentación de defensa: solo el Panel A (ratios bilaterales),
    como figura única.

    Misma lógica de cálculo:
      IS^Pasivo_{i->j,t} = e_{ij,t} / P^j_{red,t}
      Denominador: pasivo total del sector deudor j en la red
                   (= todo lo que el SF tiene colocado en j).

    Interpretación:
      IS_nb (NBFI -> Bancos): de todo el fondeo que reciben los
            bancos del SF, qué fracción proviene del NBFI.
      IS_bn (Bancos -> NBFI): de todo el fondeo que recibe el
            NBFI del SF, qué fracción proviene de los bancos.

    El hallazgo a destacar en la defensa: la dirección dominante
    es NBFI -> Bancos, atípica respecto a economías avanzadas.
    """
    print("  Fig IS Bilateral solo Panel A: NBFI<->Bancos (ratios)...")

    df_red = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df_red["t"] = df_red["periodo"].apply(tp)

    NBFI     = ["S123","S124","S125_S126","S128","S129"]
    BANCARIO = ["S122"]

    # Denominadores: pasivo total del sector deudor en la red
    pas_bancos = df_red[
        df_red["sector_pasivo_codigo"].isin(BANCARIO)
    ].groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"P_bancos"})

    pas_nbfi = df_red[
        df_red["sector_pasivo_codigo"].isin(NBFI)
    ].groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"P_nbfi"})

    # Numeradores: exposiciones bilaterales
    e_nb = df_red[
        (df_red["sector_activo_codigo"].isin(NBFI)) &
        (df_red["sector_pasivo_codigo"].isin(BANCARIO))
    ].groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"e_nbfi_bancos"})

    e_bn = df_red[
        (df_red["sector_activo_codigo"].isin(BANCARIO)) &
        (df_red["sector_pasivo_codigo"].isin(NBFI))
    ].groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"e_bancos_nbfi"})

    # IS bilaterales
    df = pas_bancos.merge(pas_nbfi, on=["periodo","t"])
    df = df.merge(e_nb, on=["periodo","t"])
    df = df.merge(e_bn, on=["periodo","t"])
    df["IS_nb"] = df["e_nbfi_bancos"] / df["P_bancos"]
    df["IS_bn"] = df["e_bancos_nbfi"] / df["P_nbfi"]
    df = df.sort_values("t")

    # Figura única
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_title(
        r"Índice de Dependencia de Fondeo Bilateral ($IS^{Pasivo}_{i \rightarrow j}$)"
        " — NBFI y Bancos agregados, 2003–2025" "\n"
        r"$IS^{Pasivo}_{i \rightarrow j,t} = e_{ij,t} / P^j_{red,t}$"
        " | denominador: pasivo total del sector deudor en la red",
        fontsize=11, pad=12
    )

    ax.plot(df["t"], df["IS_nb"]*100,
            color=COLORS["S129"], lw=2.2, ls="-",
            label="NBFI → Bancos")
    ax.plot(df["t"], df["IS_bn"]*100,
            color=COLORS["S122"], lw=2.2, ls="--",
            label="Bancos → NBFI")

    mean_nb = df["IS_nb"].mean()*100
    mean_bn = df["IS_bn"].mean()*100
    ax.axhline(mean_nb, color=COLORS["S129"], ls=":", lw=0.8, alpha=0.5)
    ax.axhline(mean_bn, color=COLORS["S122"], ls=":", lw=0.8, alpha=0.5)
    ax.text(2025.5, mean_nb+0.5, f"{mean_nb:.1f}%",
            fontsize=7, color=COLORS["S129"], va="bottom", ha="right")
    ax.text(2025.5, mean_bn+0.1, f"{mean_bn:.1f}%",
            fontsize=7, color=COLORS["S122"], va="bottom", ha="right")

    ax.set_ylabel("% del pasivo total del sector deudor en la red")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    # ylim adaptable: deja headroom sobre el máximo observado
    ymax_data = max(df["IS_nb"].max(), df["IS_bn"].max()) * 100
    ymax = max(55, int(np.ceil(ymax_data / 10) * 10 + 5))
    ax.set_ylim(0, ymax)
    ax.set_yticks(np.arange(0, ymax + 1, 10))
    xax(ax); vl(ax)
    legend_below(ax, ncol=2)

    save_fig(fig, "fig_is_bilateral_fondeo_solo_panelA")

# ============================================================
# FIG 5: HHI fuentes financiamiento sector real
# ============================================================

def fig5_hhi_fuentes():
    """
    HHI de fuentes de financiamiento del sector real.
    Panel A: Hogares — HHI (eje izq.) + participación bancaria y NBFI (eje der.)
    Panel B: Empresas NF — ídem
    Muestra que a pesar del crecimiento NBFI, la concentración no bajó.
    """
    print("  Fig 5: HHI fuentes financiamiento sector real...")

    df = pd.read_csv(DATA_DIR / "resumen_hhi_por_periodo.csv")
    df["t"] = df["periodo"].apply(tp)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        r"Concentración de fuentes de financiamiento del sector real, 2003–2025"
        "\n"
        r"($HHI_{i,t} = \sum_j s_{ij,t}^2 \times 10{,}000$ "
        "| instrumentos: préstamos + títulos de deuda)",
        fontsize=11
    )

    for ax, cod, titulo, color_hhi in [
        (axes[0], "S14_S15", "A. Hogares e IPSFLSH",        COLORS["S14_S15"]),
        (axes[1], "S11",     "B. Empresas no financieras",  COLORS["S11"]),
    ]:
        sub = df[df["sector_pasivo_codigo"] == cod].sort_values("t")

        # eje principal: HHI
        ax.plot(sub["t"], sub["hhi"],
                color=color_hhi, lw=2.2, label="HHI (eje izq.)")
        ax.set_ylabel("HHI (escala 0–10,000)", color=color_hhi)
        ax.tick_params(axis="y", labelcolor=color_hhi)
        ax.set_ylim(0, 10500)
        ax.set_yticks(np.arange(0, 11000, 2000))

        # eje secundario: participaciones
        ax2 = ax.twinx()
        #ax2.fill_between(sub["t"], sub["part_bancario"] * 100,
                         #alpha=0.12, color=COLORS["S122"])
        ax2.plot(sub["t"], sub["part_bancario"] * 100,
                 color=COLORS["S122"], lw=1.4, ls="--",
                 label="Part. bancaria % (eje der.)")
        ax2.plot(sub["t"], sub["part_nbfi"] * 100,
                 color=COLORS["NBFI"], lw=1.4, ls=":",
                 label="Part. NBFI % (eje der.)")
        ax2.set_ylabel("Participación (%)", color="gray")
        ax2.tick_params(axis="y", labelcolor="gray")
        ax2.set_ylim(0, 110)
        ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax2.spines["top"].set_visible(False)

        ax.set_title(titulo)
        xax(ax); vl(ax)

        # leyenda combinada de ambos ejes
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2,
                  loc="upper center", bbox_to_anchor=(0.5, -0.22),
                  fontsize=8, frameon=True, framealpha=0.9,
                  edgecolor="silver", ncol=3)

    save_fig(fig, "fig5_hhi_fuentes_sector_real")

# ============================================================
# FIG 6: Crédito al sector real por tipo de acreedor
# ============================================================

def fig6_credito_sector_real():
    """
    Crédito directo al sector real por tipo de acreedor.
    Panel A: Participación de cada acreedor en el crédito total (%)
    Panel B: Monto absoluto por grupo acreedor (niveles)
    Muestra que la participación NBFI en el crédito al sector real
    es estructuralmente baja pese al crecimiento del NBFI.
    """
    print("  Fig 6: Crédito al sector real por acreedor...")

    df = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df["t"] = df["periodo"].apply(tp)

    INSTR_HHI   = ["Prestamos", "Titulos deuda"]
    SECTOR_REAL = ["S11", "S14_S15"]

    cr = df[
        (df["instrumento"].isin(INSTR_HHI)) &
        (df["sector_pasivo_codigo"].isin(SECTOR_REAL))
    ].copy()

    def clasif(cod):
        if cod in ["S122"]:                                    return "Bancario"
        if cod in ["S123","S124","S125_S126","S128","S129"]:   return "NBFI"
        if cod in ["S121"]:                                    return "Banco Central"
        if cod in ["S13"]:                                     return "Gobierno"
        if cod in ["S2"]:                                      return "Resto del mundo"
        return "Otros"

    cr["grupo"] = cr["sector_activo_codigo"].apply(clasif)

    agg = cr.groupby(["periodo","t","grupo"])["valor"].sum().reset_index()
    tot = agg.groupby(["periodo","t"])["valor"].sum().reset_index().rename(
        columns={"valor":"total"})
    agg = agg.merge(tot, on=["periodo","t"])
    agg["part"] = agg["valor"] / agg["total"] * 100

    GRUPOS = [
        ("Bancario",        COLORS["S122"], "--",  2.0),
        ("NBFI",            COLORS["NBFI"], ":", 2.0),
        ("Gobierno",        COLORS["S13"],  "--", 1.4),
        ("Resto del mundo", COLORS["S121"],   "-",  1.2),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(
        "Crédito directo al sector real por tipo de acreedor, 2003–2025\n"
        "(instrumentos: préstamos + títulos de deuda)",
        fontsize=11
    )

    # ── Panel A: participaciones ──────────────────────────────────────
    ax = axes[0]
    for grp, col, ls, lw in GRUPOS:
        sub = agg[agg["grupo"] == grp].sort_values("t")
        if sub.empty: continue
        ax.plot(sub["t"], sub["part"],
                color=col, lw=lw, ls=ls, label=grp)

    ax.set_title("A. Participación en el crédito total al sector real")
    ax.set_ylabel("% del crédito total")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylim(0, 100)
    xax(ax); vl(ax)
    legend_below(ax, ncol=2)

    # ── Panel B: niveles ──────────────────────────────────────────────
    ax2 = axes[1]
    for grp, col, ls, lw in GRUPOS:
        sub = agg[agg["grupo"] == grp].sort_values("t")
        if sub.empty: continue
        ax2.plot(sub["t"], sub["valor"] / 1000,
                 color=col, lw=lw, ls=ls, label=grp)

    ax2.set_title("B. Monto de crédito por acreedor (billones CLP)")
    ax2.set_ylabel("Miles de MMCL (billones CLP)")
    xax(ax2); vl(ax2)
    legend_below(ax2, ncol=2)

    save_fig(fig, "fig6_credito_sector_real")

# ============================================================
# FIG 7: PASIVOS DEL SECTOR REAL POR CONTRAPARTE ACREEDORA
# ============================================================

def fig7_pasivos_sector_real():
    """
    Pasivos del sector real (Hogares y Empresas NF) en la red
    who-to-whom, identificando la contraparte acreedora:
    - Bancos (S122)
    - AFP (S129)
    - Seguros (S128)
    - FMM (S123)
    - FNM (S124)
    - Otros intermediarios (S125_S126)
    - Gobierno (S13)
    - Resto del mundo (S2)
    - Banco Central (S121)

    Panel A: Hogares — ratio sobre pasivo total en red (%)
    Panel B: Empresas NF — ratio sobre pasivo total en red (%)
    Panel C: Hogares — niveles (billones CLP)
    Panel D: Empresas NF — niveles (billones CLP)

    Instrumentos: Préstamos + Títulos de deuda
    (únicos pasivos del sector real en la red)
    """
    print("  Fig X: Pasivos sector real por contraparte acreedora...")

    df = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df["t"] = df["periodo"].apply(tp)

    INSTR_HHI   = ["Prestamos", "Titulos deuda"]
    SECTOR_REAL = ["S11", "S14_S15"]

    # Filtrar crédito directo al sector real
    cr = df[
        (df["instrumento"].isin(INSTR_HHI)) &
        (df["sector_pasivo_codigo"].isin(SECTOR_REAL))
    ].copy()

    # Clasificar acreedores — desagregado
    ACREEDORES = {
        "S122":      ("Bancos",           COLORS["S122"], "-",  2.2),
        "S129":      ("AFP",              COLORS["S129"], "-",  1.8),
        "S128":      ("Seguros",          COLORS["S128"], "--", 1.6),
        "S124":      ("FNM",              COLORS["S124"], "--", 1.6),
        "S123":      ("FMM",              COLORS["S123"], ":",  1.4),
        "S125_S126": ("Otros interm.",    COLORS["S125_S126"], ":", 1.4),
        "S13":       ("Gobierno",         COLORS["S13"],  "--", 1.2),
        "S2":        ("Resto del mundo",  COLORS["S2"],   "--",  1.2),
        "S121":      ("Banco Central",    COLORS["S121"], "--", 1.0),
    }

    # Agregar por período × sector pasivo × acreedor
    agg = cr.groupby(
        ["periodo","t","sector_pasivo_codigo","sector_activo_codigo"]
    )["valor"].sum().reset_index()

    # Total pasivo por sector real × período
    tot = agg.groupby(
        ["periodo","t","sector_pasivo_codigo"]
    )["valor"].sum().reset_index().rename(columns={"valor":"total"})
    agg = agg.merge(tot, on=["periodo","t","sector_pasivo_codigo"])
    agg["pct"] = agg["valor"] / agg["total"] * 100

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    fig.suptitle(
        "Pasivos del sector real por contraparte acreedora, 2003–2025\n"
        "(instrumentos: préstamos + títulos de deuda | "
        "denominador: pasivo total del sector en la red who-to-whom)",
        fontsize=11
    )

    PANELES = [
        ("S14_S15", "A. Hogares — participación (%)",      axes[0][0], "pct"),
        ("S11",     "B. Empresas NF — participación (%)",  axes[0][1], "pct"),
        ("S14_S15", "C. Hogares — niveles (billones CLP)", axes[1][0], "valor"),
        ("S11",     "D. Empresas NF — niveles (billones CLP)", axes[1][1], "valor"),
    ]

    for cod_deudor, titulo, ax, metrica in PANELES:
        sub_d = agg[agg["sector_pasivo_codigo"] == cod_deudor]

        for cod_ac, (label, color, ls, lw) in ACREEDORES.items():
            sub = sub_d[
                sub_d["sector_activo_codigo"] == cod_ac
            ].sort_values("t")
            if sub.empty or sub[metrica].sum() == 0:
                continue
            y = sub[metrica] if metrica == "pct" else sub[metrica] / 1000
            ax.plot(sub["t"], y,
                    color=color, lw=lw, ls=ls, label=label)

        ax.set_title(titulo)
        xax(ax); vl(ax)

        if metrica == "pct":
            ax.set_ylabel("% del pasivo total en red")
            ax.yaxis.set_major_formatter(mtick.PercentFormatter())
            ax.set_ylim(0, 105)
        else:
            ax.set_ylabel("Miles de MMCL (billones CLP)")
            ax.set_yticks(np.arange(0, 150, 20))

     # Eje secundario con HHI — solo en paneles de participación
        if metrica == "pct":
            df_hhi = pd.read_csv(DATA_DIR / "resumen_hhi_por_periodo.csv")
            df_hhi["t"] = df_hhi["periodo"].apply(tp)
            sub_hhi = df_hhi[
                df_hhi["sector_pasivo_codigo"] == cod_deudor
            ].sort_values("t")

            ax_r = ax.twinx()
            ax_r.plot(sub_hhi["t"], sub_hhi["hhi"],
                      color="black", lw=3, ls="-.",
                      alpha=0.7, label="HHI (eje der.)")
            ax_r.set_ylabel("HHI (escala 0–10,000)", color="black",
                            fontsize=8)
            ax_r.tick_params(axis="y", labelcolor="black")
            ax_r.set_ylim(0, 11000)
            ax_r.set_yticks([0, 2000, 4000, 6000, 8000, 10000])
            ax_r.spines["top"].set_visible(False)

            # Añadir HHI a la leyenda combinada
            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax_r.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2,
                      loc="upper center",
                      bbox_to_anchor=(0.5, -0.22),
                      fontsize=8, frameon=True,
                      framealpha=0.9, edgecolor="silver",
                      ncol=3)

    save_fig(fig, "fig7_pasivos_sector_real")

# ============================================================
# FIG 8: PASIVOS DEL SECTOR REAL — BANCARIO vs NBFI vs OTROS
# ============================================================

def fig8_pasivos_sector_real_agregado():
    """
    Mismos 4 paneles que figX pero con NBFI agregado
    (S123+S124+S125_S126+S128+S129) en lugar de desagregado.

    Grupos:
    - Bancos (S122)
    - NBFI agregado (S123+S124+S125_S126+S128+S129)
    - Gobierno (S13)
    - Resto del mundo (S2)
    - Banco Central (S121)

    Permite ver de forma limpia la asimetría Bancos vs NBFI
    en el financiamiento del sector real.
    """
    print("  Fig Y: Pasivos sector real — Bancario vs NBFI agregado...")

    df = pd.read_csv(DATA_DIR / "panel_red_saldos.csv")
    df["t"] = df["periodo"].apply(tp)

    INSTR_HHI   = ["Prestamos", "Titulos deuda"]
    SECTOR_REAL = ["S11", "S14_S15"]
    NBFI        = ["S123","S124","S125_S126","S128","S129"]

    cr = df[
        (df["instrumento"].isin(INSTR_HHI)) &
        (df["sector_pasivo_codigo"].isin(SECTOR_REAL))
    ].copy()

    # Clasificar en grupos agregados
    def grupo(cod):
        if cod == "S122":           return "Bancos"
        if cod in NBFI:             return "NBFI agregado"
        if cod == "S13":            return "Gobierno"
        if cod == "S2":             return "Resto del mundo"
        if cod == "S121":           return "Banco Central"
        return "Otros"

    cr["grupo"] = cr["sector_activo_codigo"].apply(grupo)

    # Agregar por período × sector pasivo × grupo
    agg = cr.groupby(
        ["periodo","t","sector_pasivo_codigo","grupo"]
    )["valor"].sum().reset_index()

    tot = agg.groupby(
        ["periodo","t","sector_pasivo_codigo"]
    )["valor"].sum().reset_index().rename(columns={"valor":"total"})
    agg = agg.merge(tot, on=["periodo","t","sector_pasivo_codigo"])
    agg["pct"] = agg["valor"] / agg["total"] * 100

    GRUPOS = {
        "Bancos":          (COLORS["S122"],      "-",  2.2),
        "NBFI agregado":   (COLORS["S129"],      "-",  2.2),
        "Gobierno":        (COLORS["S13"],       "--", 1.4),
        "Resto del mundo": (COLORS["S2"],        "--",  1.4),
        "Banco Central":   (COLORS["S121"],      "--", 1.0),
    }

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    fig.suptitle(
        "Pasivos del sector real por grupo acreedor, 2003–2025\n"
        "(instrumentos: préstamos + títulos de deuda | "
        "denominador: pasivo total del sector en la red who-to-whom)",
        fontsize=11
    )

    PANELES = [
        ("S14_S15", "A. Hogares — participación (%)",
         axes[0][0], "pct"),
        ("S11",     "B. Empresas NF — participación (%)",
         axes[0][1], "pct"),
        ("S14_S15", "C. Hogares — niveles (billones CLP)",
         axes[1][0], "valor"),
        ("S11",     "D. Empresas NF — niveles (billones CLP)",
         axes[1][1], "valor"),
    ]

    for cod_deudor, titulo, ax, metrica in PANELES:
        sub_d = agg[agg["sector_pasivo_codigo"] == cod_deudor]

        for grp, (color, ls, lw) in GRUPOS.items():
            sub = sub_d[sub_d["grupo"] == grp].sort_values("t")
            if sub.empty or sub[metrica].sum() == 0:
                continue
            y = sub[metrica] if metrica == "pct" \
                else sub[metrica] / 1000
            ax.plot(sub["t"], y,
                    color=color, lw=lw, ls=ls, label=grp)

        ax.set_title(titulo)
        xax(ax); vl(ax)

        if metrica == "pct":
            ax.set_ylabel("% del pasivo total en red")
            ax.yaxis.set_major_formatter(mtick.PercentFormatter())
            ax.set_ylim(0, 105)
        else:
            ax.set_ylabel("Miles de MMCL (billones CLP)")

             # Eje secundario con HHI — solo en paneles de participación
        if metrica == "pct":
            df_hhi = pd.read_csv(DATA_DIR / "resumen_hhi_por_periodo.csv")
            df_hhi["t"] = df_hhi["periodo"].apply(tp)
            sub_hhi = df_hhi[
                df_hhi["sector_pasivo_codigo"] == cod_deudor
            ].sort_values("t")

            ax_r = ax.twinx()
            ax_r.plot(sub_hhi["t"], sub_hhi["hhi"],
                      color="black", lw=3, ls="-.",
                      alpha=0.7, label="HHI (eje der.)")
            ax_r.set_ylabel("HHI (escala 0–10,000)", color="black",
                            fontsize=8)
            ax_r.tick_params(axis="y", labelcolor="black")
            ax_r.set_ylim(0, 11000)
            ax_r.set_yticks([0, 2000, 4000, 6000, 8000, 10000])
            ax_r.spines["top"].set_visible(False)

            # Añadir HHI a la leyenda combinada
            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax_r.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2,
                      loc="upper center",
                      bbox_to_anchor=(0.5, -0.22),
                      fontsize=8, frameon=True,
                      framealpha=0.9, edgecolor="silver",
                      ncol=3)

    save_fig(fig, "fig8_pasivos_sector_real_agregado")

# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

def main():
    print("\n" + "=" * 60)
    print("ANÁLISIS DESCRIPTIVO — PARTE 1 TESIS")
    print("=" * 60)

    for f in ["panel_balances.csv", "panel_matriz_saldos.csv"]:
        if not (DATA_DIR / f).exists():
            print(f"\n⚠️  No se encontró: {DATA_DIR / f}")
            print("   Verifica que DATA_DIR apunte a la carpeta correcta.")
            return

    print("\nCargando datos...")
    bal = pd.read_csv(DATA_DIR / "panel_balances.csv")
    mat = pd.read_csv(DATA_DIR / "panel_matriz_saldos.csv")
    bal["t"] = bal["periodo"].apply(tp)
    mat["t"] = mat["periodo"].apply(tp)
    print(f"  ✓ Balances: {bal.shape}  |  Matriz saldos: {mat.shape}")

    print("\nGenerando figuras...")
    fig1_activos_por_sector(bal)
    fig2_activos_pib(bal)
    fig2b_activos_red_pib()
    fig2b_activos_red_pib_1x2()  
    fig2c_composicion_instrumentos()
    fig2c_simple_composicion_instrumentos()
    fig2d_composicion_pasivos()

    fig3_exposicion_bilateral_asimetria()   
    fig4_exposicion_sector_nbfi()
    fig_is_bilateral_fondeo()
    fig_is_bilateral_fondeo_agregado()
    fig_is_bilateral_fondeo_solo_panelA()
    
    fig5_hhi_fuentes()
    fig6_credito_sector_real()
    fig7_pasivos_sector_real()
    fig8_pasivos_sector_real_agregado()

    print("\n" + "=" * 60)
    print("OUTPUTS GENERADOS")
    print("=" * 60)
    print(f"  Figuras (PNG) → {FIG_DIR}")
    print("\n✅ Análisis Parte 1 completado.\n")


if __name__ == "__main__":
    main()



  Fig 3.C: Rol del BC durante pandemia y retiros...
    ✓ fig_3C_bc_rol_pandemia

ANÁLISIS DESCRIPTIVO — PARTE 1 TESIS

Cargando datos...
  ✓ Balances: (1001, 37)  |  Matriz saldos: (13792, 11)

Generando figuras...
  Fig 1: Activos por sector...
    ✓ fig1_1_activos_sector
  Fig 11: Activos como % del PIB (suma móvil 4 trimestres)...
    ✓ fig2_activos_pib
  Fig 2B: Activos desde red who-to-whom como % del PIB...
    ✓ fig2b_activos_red_pib_2x1
  Fig 2B: Activos desde red who-to-whom como % del PIB...
    ✓ fig2b_activos_red_pib_1x2
  Fig 2C: Composición de activos por sector (desde red)...
    ✓ fig2c_composicion_instrumentos
  Fig 2C simple: Composición por sector (6 sectores, defensa)...
    ✓ fig2c_simple_composicion_instrumentos
  Fig 2D: Composición de pasivos por sector (desde red)...
    ✓ fig2d_composicion_pasivos
  Fig 3: Asimetría exposición bilateral NBFI↔Bancos...
    ✓ fig3_exposicion_asimetria
  Fig 4: Exposición por sector NBFI hacia bancos...
    ✓ fig4_exposicion_sec

In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../1_datos/1_clean_data")

df = pd.read_csv(DATA_DIR / "panel_exp_nbfi_bancos_agregado.csv")
print(df.columns.tolist())
print(df.head(3))

['periodo', 'año', 'trimestre', 'activo_nbfi_red_total', 'ratio_exp_nbfi_bancos_agg']
  periodo   año  trimestre  activo_nbfi_red_total  ratio_exp_nbfi_bancos_agg
0  2003T1  2003          1           40002.255311                   0.403557
1  2003T2  2003          2           42014.291950                   0.391827
2  2003T3  2003          3           42820.452629                   0.376258
